In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. LOAD THE DATA 

In [2]:
print("Loading data and matrices...")
df = pd.read_csv('data/processed/cleaned_retail_data_2022_2025.csv')
user_item_matrix = pd.read_pickle('data/processed/user_item_matrix.pkl')

Loading data and matrices...


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/cleaned_retail_data_2022_2025.csv'

# 2. POPULARITY BASELINE

In [ ]:
# In retail, "popularity" is usually better measured by how MANY times an item 
# was bought (Frequency) rather than total quantity, to avoid bulk-buy skew.
def train_popularity_model(data, top_n=10):
    """
    Calculates the most popular products based on the number of unique invoices.
    """
    popularity = data.groupby('ProductName')['InvoiceID'].nunique().reset_index()
    popularity.columns = ['ProductName', 'PurchaseFrequency']
    popularity = popularity.sort_values(by='PurchaseFrequency', ascending=False)
    
    return popularity.head(top_n)

top_popular_items = train_popularity_model(df, top_n=10)
print("\n--- POPULARITY BASELINE ---")
print("Top 5 Most Popular Products:")
print(top_popular_items.head(5).to_string(index=False))


# 3. ITEM-BASED COLLABORATIVE FILTERING

In [ ]:
# We calculate the Cosine Similarity between items. 
# If Item A and Item B are frequently bought by the same contractors, 
# their cosine similarity score will be close to 1.

print("\nCalculating Item Similarity Matrix (this might take a moment)...")
# Transpose the matrix so products are rows and customers are columns
item_user_matrix = user_item_matrix.T 

# Calculate cosine similarity between items
item_sim_array = cosine_similarity(item_user_matrix)

# Convert the resulting array back into a readable DataFrame
item_similarity_df = pd.DataFrame(
    item_sim_array, 
    index=user_item_matrix.columns, 
    columns=user_item_matrix.columns
)

def get_item_similarity_recommendations(product_name, similarity_matrix, top_n=5):
    """
    Given a product name, returns the top N most similar products.
    """
    if product_name not in similarity_matrix.columns:
        return f"Product '{product_name}' not found in the matrix."
    
    # Get similarity scores for the target product
    similar_scores = similarity_matrix[product_name]
    
    # Sort scores in descending order, drop the product itself (similarity = 1.0)
    similar_items = similar_scores.sort_values(ascending=False).drop(product_name)
    
    return similar_items.head(top_n)

# Example Usage: Let's pick a specific product from your data
# (Replace with an actual product name from your dataset to test)
sample_product = user_item_matrix.columns[0] # Just picking the first product alphabetically
print(f"\n--- ITEM SIMILARITY MODEL ---")
print(f"Because customers bought: {sample_product}")
print("They also bought:")
print(get_item_similarity_recommendations(sample_product, item_similarity_df, top_n=5))



# 4. EXPORT

In [ ]:
# Save the similarity matrix for the Hybrid Model later
item_similarity_df.to_pickle('data/processed/item_similarity_matrix.pkl')